# 🧠 Delentia SLM — JITNA v0.1 QLoRA Fine-tuning

**Model**: Llama 3.1 8B → QLoRA fine-tune → GGUF Q4_K_M export  
**Target metrics**: JITNA ≥ 94% · FDIA ≥ 0.87 · Hallucination ≤ 2.8%  
**GPU**: T4 (free) → ~4–6h | A100 (Colab Pro) → ~1.5h

---
### Steps
1. Mount Drive + clone repo
2. Install dependencies
3. Extract dataset from delentia-os
4. Validate dataset (gate: ≥500 pairs, avg FDIA ≥ 0.7)
5. Fine-tune (QLoRA)
6. Evaluate (gate: all metrics pass)
7. Export GGUF Q4_K_M
8. Upload to HuggingFace Hub
9. Smoke-test with Ollama

In [ ]:
# ─── Cell 1: Mount Google Drive + clone repo ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

REPO_URL = 'https://github.com/delentia-labs/delentia-ai.git'
REPO_DIR = '/content/delentia-ai'

if not os.path.exists(REPO_DIR):
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print('Repo already exists — pulled latest:', result.stdout.strip())

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ─── Cell 2: Install dependencies ────────────────────────────────────────────
# Unsloth optimized for T4/A100 — 2x faster training, 60% less VRAM
import subprocess, sys

# Detect GPU type for optimal install
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                          capture_output=True, text=True).stdout.strip()
print(f'GPU detected: {gpu_info}')

# Install Unsloth + training deps
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git',
    '--quiet'
])

# Install project requirements
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '--quiet'
])

print('✅ All dependencies installed')

In [ ]:
# ─── Cell 3: Extract dataset from delentia-os ─────────────────────────────────
# Pulls JITNA instruction pairs from the OS benchmark + contracts
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'datasets/scripts/extract_from_os.py',
     '--toon'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Dataset extraction failed')

# Show sample
import json
with open('datasets/processed/jitna_pairs_toon.jsonl') as f:
    lines = f.readlines()
print(f'\n📊 Total pairs extracted: {len(lines)}')
print('Sample pair:')
print(json.dumps(json.loads(lines[0]), indent=2, ensure_ascii=False))

In [ ]:
# ─── Cell 4: Validate dataset (GATE) ─────────────────────────────────────────
# Gate: ≥500 pairs AND average FDIA ≥ 0.70
# Training will NOT proceed if either condition fails.
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'datasets/scripts/validate_dataset.py',
     'datasets/processed/jitna_pairs_toon.jsonl',
     '--toon'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('VALIDATION FAILED — STOPPING')
    print(result.stderr)
    raise SystemExit('Dataset validation gate failed. Add more data before training.')

print('✅ Dataset validation PASSED — proceeding to training')

In [ ]:
# ─── Cell 5: QLoRA Fine-tuning ────────────────────────────────────────────────
# Llama 3.1 8B → QLoRA r=16, alpha=32, target_modules=all-linear
# T4: ~4–6h | A100: ~1.5h
# Checkpoints saved to: models/checkpoints/
# Best model saved to:  models/finetuned/
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'training/finetune.py',
     '--config', 'training/config/slm_jitna_v0.2.yaml',
     '--toon'],
    # Stream stdout live
    stdout=None, stderr=None
)
if result.returncode != 0:
    raise RuntimeError('Fine-tuning failed — check logs above')

print('\n✅ Fine-tuning complete — model saved to models/finetuned/')

In [ ]:
# ─── Cell 6: Evaluate (GATE) ─────────────────────────────────────────────────
# Gate: JITNA ≥ 94% · FDIA ≥ 0.87 · Hallucination ≤ 2.8%
# Export will NOT proceed if any gate fails.
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'training/evaluate.py',
     '--config', 'training/config/slm_jitna_v0.2.yaml',
     '--toon'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('EVALUATION GATE FAILED — NOT exporting')
    print(result.stderr)
    raise SystemExit('Model did not meet minimum quality gates. Retrain with more data or longer epochs.')

print('✅ All quality gates PASSED — proceeding to GGUF export')

In [ ]:
# ─── Cell 7: Export to GGUF Q4_K_M ───────────────────────────────────────────
# Quantizes the fine-tuned model → GGUF format for Ollama + llama.cpp
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'training/export_gguf.py',
     '--toon'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('GGUF export failed')

import os, glob
gguf_files = glob.glob('models/gguf/*.gguf')
print(f'\n✅ GGUF files created:')
for f in gguf_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {f} ({size_mb:.0f} MB)')

In [ ]:
# ─── Cell 8: Upload to HuggingFace Hub ───────────────────────────────────────
# Requires: HF_TOKEN secret in Colab Secrets (key icon in left sidebar)
import os, glob
from huggingface_hub import login, HfApi, ModelCard, ModelCardData

# Read token from Colab secrets (recommended) or environment variable
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')

if not hf_token:
    raise ValueError('HF_TOKEN not set. Add it in Colab Secrets (key icon) before running.')

login(token=hf_token)

REPO_ID = 'delentia-labs/delentia-slm-jitna-v0.2'
api = HfApi()

# Create repo if it doesn't exist
api.create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True, private=False)

# Upload GGUF files (with progress)
gguf_files = glob.glob('models/gguf/*.gguf')
if not gguf_files:
    print('⚠️  No GGUF files found — run Cell 7 first')
else:
    for gguf_file in gguf_files:
        filename = os.path.basename(gguf_file)
        size_mb = os.path.getsize(gguf_file) / 1024 / 1024
        print(f'Uploading {filename} ({size_mb:.0f} MB)...')
        api.upload_file(
            path_or_fileobj=gguf_file,
            path_in_repo=f'gguf/{filename}',
            repo_id=REPO_ID,
            repo_type='model',
        )
        print(f'  ✅ https://huggingface.co/{REPO_ID}/blob/main/gguf/{filename}')

# Upload model card (README_MODEL.md → README.md on HF Hub)
if os.path.exists('README_MODEL.md'):
    print('\nUploading model card...')
    api.upload_file(
        path_or_fileobj='README_MODEL.md',
        path_in_repo='README.md',
        repo_id=REPO_ID,
        repo_type='model',
    )
    print('  ✅ Model card uploaded')
else:
    print('⚠️  README_MODEL.md not found — model card not uploaded')

# Upload training config for reproducibility
for config_file in glob.glob('training/config/*.yaml'):
    fname = os.path.basename(config_file)
    api.upload_file(
        path_or_fileobj=config_file,
        path_in_repo=f'training_config/{fname}',
        repo_id=REPO_ID,
        repo_type='model',
    )
    print(f'  ✅ Config uploaded: training_config/{fname}')

print(f'\n✅ Model fully published at: https://huggingface.co/{REPO_ID}')
print(f'   Use with Ollama: ollama run delentia-labs/delentia-jitna-v0.2')
print(f'   Download GGUF:   huggingface-cli download {REPO_ID} gguf/delentia-jitna-v0.2-Q4_K_M.gguf')

In [ ]:
# ─── Cell 9: Smoke-test with Ollama ──────────────────────────────────────────
# Installs Ollama in Colab, creates a Modelfile, runs one inference.
import subprocess, os, glob

# Install Ollama
result = subprocess.run(
    'curl -fsSL https://ollama.com/install.sh | sh',
    shell=True, capture_output=True, text=True
)
print('Ollama install:', result.returncode)

# Find the GGUF file
gguf_files = glob.glob('models/gguf/*.gguf')
if not gguf_files:
    raise FileNotFoundError('No .gguf files found — run Cell 7 first')
gguf_path = os.path.abspath(gguf_files[0])

# Create Modelfile
modelfile_content = f'''FROM {gguf_path}
PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER stop "<|eot_id|>"
SYSTEM """You are Delentia OS v0.2 — a constitutional AI operating under RCT v5 governance. You process intents through the JITNA v3 protocol. You respond in TOON format (Token-Oriented Object Notation) for token efficiency. Your responses must be factual, safe, and PDPA-compliant. You must respond using the 6 JITNA fields: I=Intent, D=Data, Δ=Delta, A=Approach, R=Reflection, M=Memory."""
'''

with open('/tmp/Modelfile', 'w') as f:
    f.write(modelfile_content)

# Start Ollama server in background
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import time; time.sleep(3)

# Create + run model
subprocess.run(['ollama', 'create', 'delentia-jitna-v0.2', '-f', '/tmp/Modelfile'], check=True)

result = subprocess.run(
    ['ollama', 'run', 'delentia-jitna-v0.2',
     'สรุปหลักการ RCT v5 HexaCore ใน 2 ประโยค'],
    capture_output=True, text=True, timeout=120
)
print('\n🤖 Model response:')
print(result.stdout)
print('\n✅ Smoke test complete!')